# Benchmark 1: Pitchfork Bifurcation

Numerical results of PCE-Galerkin on the canonical supercritical pitchfork normal form:
$$\dot{u} = f(u, \mu) = -u^3 + \mu u$$

This notebook reproduces:
1. **Oscillating and branch-approximating roots**
2. **Convergence rates** of the upper branch $\bar{u}_+(\mu) = \sqrt{\mu}\,\chi_{[0,\infty)}(\mu)$ across three regularity regimes:
   - **Smooth regime:** $I = [0.2, 1]$.
   - **Boundary singularity:** $I = [0, 1]$.
   - **Interior singularity:** $I = [-1, 3]$.

In [1]:
import sys
sys.path.append("..")  # Permette di importare pce_bifurcation dalla root della repo

import numpy as np
import matplotlib.pyplot as plt
from pce_galerkin import (
    PCEGalerkinSolver,
    set_plot_style,
    plot_branches_2d,
    plot_convergence,
)

set_plot_style(usetex=True)

# Campo vettoriale e Jacobiano (u ha shape (1, Q), mu ha shape (Q,))
def rhs_pitchfork(u, mu):
    return -u**3 + mu[None, :] * u

def jac_pitchfork(u, mu):
    return (-3.0 * u**2 + mu[None, :])[None, :, :]

# Ramo analitico superiore u_+(mu)
def exact_upper_branch(mu):
    return np.sqrt(np.maximum(mu, 0.0))[None, :]

## 1. Oscillating Solutions vs. Branch-Approximating Solutions (Figure 1)

Solving the nonlinear Galerkin system directly at $N = 10$ from random initializations produces a large family of spurious oscillatory roots, whereas **Algorithm 1 (degree continuation)** isolates the three physical branches on $I = [0, 2]$.

In [2]:
solver_fig1 = PCEGalerkinSolver(
    rhs=rhs_pitchfork,
    jac_u=jac_pitchfork,
    mu_bounds=(0.0, 2.0),
    state_dim=1,
    quadrature_order=60,
)

# 1. Risoluzione diretta a N=10 senza continuation (radici spurie + fisiche)
all_roots = solver_fig1.direct_random_roots(degree=10, n_samples=30, seed=1, scale=0.8)

# 2. Algorithm 1: degree continuation da N=0 a N=10 (solo rami fisici)
physical_branches = solver_fig1.find_multiple_branches(max_degree=10, n_samples=30, seed=42, unique_tol=1e-6, box=(-1.5, 1.5))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)

# Pannello sinistro: tutte le radici algebriche
plot_branches_2d(solver_fig1, all_roots, ax=axes[0])
axes[0].set_title(rf"Direct Galerkin roots ($N=10$, found {len(all_roots)})")
axes[0].set_ylim([-2.0, 2.0])

# Pannello destro: rami isolati con Algorithm 1 + rami esatti sovrapposti
mu_ex = np.linspace(0.0, 2.0, 400)
for b_ex in [np.sqrt(mu_ex), np.zeros_like(mu_ex), -np.sqrt(mu_ex)]:
    axes[1].plot(mu_ex, b_ex, "k:", lw=3.0, zorder=5)

plot_branches_2d(solver_fig1, physical_branches, ax=axes[1], ylabel="")
axes[1].set_title(rf"Branch-approximating ($N=10$, found {len(physical_branches)})")

plt.tight_layout()
plt.show()

RuntimeError: Failed to process string with tex because latex could not be found

Error in callback <function _draw_all_if_interactive at 0x7ff70ed4ea20> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 1100x420 with 2 Axes>